Importing pandas and opening a dataset

In [1]:
import sqlalchemy
import psycopg2
import pandas as pd

print(sqlalchemy.__version__)


2.0.46


In [2]:
import sys, site
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Prefix:", sys.prefix)
print("Site-packages:", site.getsitepackages() if hasattr(site, "getsitepackages") else "n/a")


Python: 3.12.12 (main, Dec  9 2025, 02:06:02) [GCC 12.2.0]
Executable: /usr/local/bin/python
Prefix: /usr/local
Site-packages: ['/usr/local/lib/python3.12/site-packages']


In [3]:
import sys
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install sqlalchemy psycopg2-binary


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [4]:
import sqlalchemy, psycopg2
print(sqlalchemy.__version__)


2.0.46


In [5]:
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine(
    "postgresql+psycopg2://hi5304_user:hi5304@db:5432/hi5304"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT 1")).scalar())








1


In [6]:
cardio10 = pd.read_sql(
    "SELECT * FROM cardio10 LIMIT 20;",
    engine
)

cardio10


,id,age_days,sex,height,weight,sbp,diastolic,cholesterol,gluc,smoking,...,dm,egfr,bptreat,statin,uacr,sdi,hba1c,prevent_full_10yr_cvd,prevent_full_10yr_ascvd,prevent_full_10yr_hf
0,57792,14455,0,164,62,110,70,0,0,0,...,0,88,1,1,0.5,2,6.5,0.199410,0.657339,0.199410
1,17274,15452,0,156,48,110,70,0,0,0,...,0,80,0,1,1.0,1,5.4,0.208530,0.762402,0.208530
2,59931,15235,1,168,72,110,80,1,0,0,...,0,109,0,1,6.7,1,5.6,0.212875,0.828295,0.212875
3,72971,15370,1,178,78,110,70,0,0,0,...,0,100,0,1,4.8,1,5.6,0.215769,0.540801,0.215769
4,33519,15184,0,153,63,110,70,0,0,0,...,0,81,0,0,2.2,4,5.1,0.217907,0.323234,0.217907
5,52147,15354,1,171,77,120,80,0,0,0,...,0,119,0,1,1.5,1,6.4,0.218856,0.447099,0.218856
6,35807,14635,1,168,68,120,80,0,0,0,...,0,76,0,1,3.7,1,5.3,0.221457,0.255052,0.221457
7,64620,14542,0,155,67,110,70,0,0,0,...,0,65,0,0,6.0,6,5.9,0.249106,0.248927,0.249106
8,79749,10964,0,160,59,110,70,0,0,0,...,0,95,1,1,174.4,4,5.3,0.252480,0.523174,0.252480
9,72254,15145,1,160,70,120,80,0,0,0,...,0,89,0,0,11.4,1,4.5,0.257121,0.581944,0.257121


In [7]:
patients = pd.read_sql("SELECT * FROM patients ORDER BY patient_id;", engine)
patients.head()



,patient_id,first_name,last_name,date_of_birth,sex,race,ethnicity,zip_code
0,1001,John,Doe,1968-04-12,M,White,Non-Hispanic,75201
1,1002,Maria,Lopez,1975-09-30,F,Hispanic,Hispanic,75204
2,1003,James,Smith,1982-01-18,M,Black,Non-Hispanic,75080
3,1004,Linda,Chen,1990-06-05,F,Asian,Non-Hispanic,75024
4,1005,Robert,Johnson,1959-11-22,M,White,Non-Hispanic,75230


In [8]:

medications = pd.read_sql("SELECT * FROM medications;", engine)
medications.head()


,medication_id,patient_id,medication,dose,frequency,date_started,indication
0,1,1001,Lisinopril,10 mg,Once daily,2024-09-01,Hypertension
1,2,1002,Amlodipine,5 mg,Once daily,2024-08-20,Hypertension
2,3,1003,Metoprolol,50 mg,Twice daily,2024-09-10,Hypertension
3,4,1003,Atorvastatin,20 mg,Once daily,2024-09-10,Hyperlipidemia
4,5,1005,Losartan,50 mg,Once daily,2024-09-05,Hypertension


In [9]:
bp = pd.read_sql("SELECT * FROM bp_readings ORDER BY bp_id;", engine)
bp.head()



,bp_id,patient_id,reading_date,systolic,diastolic,heart_rate,source
0,1,1001,2024-09-01,148,92,78,Clinic
1,2,1001,2024-10-01,142,88,76,Home
2,3,1001,2024-11-01,136,84,74,Home
3,4,1002,2024-09-15,132,82,70,Clinic
4,5,1002,2024-10-15,128,80,68,Home


In [10]:
query = """
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  b.reading_date,
  b.systolic,
  b.diastolic,
  b.heart_rate,
  b.source,
  m.medication,
  m.dose,
  m.frequency
FROM patients p
LEFT JOIN bp_readings b ON p.patient_id = b.patient_id
LEFT JOIN medications m ON p.patient_id = m.patient_id
ORDER BY p.patient_id, b.reading_date;
"""
df = pd.read_sql(query, engine)
df.head(20)


,patient_id,first_name,last_name,reading_date,systolic,diastolic,heart_rate,source,medication,dose,frequency
0,1001,John,Doe,2024-09-01,148,92,78,Clinic,Lisinopril,10 mg,Once daily
1,1001,John,Doe,2024-10-01,142,88,76,Home,Lisinopril,10 mg,Once daily
2,1001,John,Doe,2024-11-01,136,84,74,Home,Lisinopril,10 mg,Once daily
3,1002,Maria,Lopez,2024-09-15,132,82,70,Clinic,Amlodipine,5 mg,Once daily
4,1002,Maria,Lopez,2024-10-15,128,80,68,Home,Amlodipine,5 mg,Once daily
5,1003,James,Smith,2024-09-10,150,96,82,Clinic,Atorvastatin,20 mg,Once daily
6,1003,James,Smith,2024-09-10,150,96,82,Clinic,Metoprolol,50 mg,Twice daily
7,1003,James,Smith,2024-10-10,146,94,80,Home,Atorvastatin,20 mg,Once daily
8,1003,James,Smith,2024-10-10,146,94,80,Home,Metoprolol,50 mg,Twice daily
9,1004,Linda,Chen,2024-09-20,118,76,66,Clinic,NaN,NaN,NaN


In [11]:
%pip install requests


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [12]:
import requests
requests.__version__


'2.32.5'

In [13]:
import requests

url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks"
r = requests.get(url, headers={"User-Agent": "hi5304"})
r.raise_for_status()

data = r.json()
data["full_name"], data["open_issues_count"]


('Health-informatics-Data-Analytics/hi5304-notebooks', 0)

In [14]:
import json
print(json.dumps(data, indent=2)[:1500])  # prints the first ~1500 characters nicely


{
  "id": 1129982692,
  "node_id": "R_kgDOQ1oq5A",
  "name": "hi5304-notebooks",
  "full_name": "Health-informatics-Data-Analytics/hi5304-notebooks",
  "private": false,
  "owner": {
    "login": "Health-informatics-Data-Analytics",
    "id": 253527654,
    "node_id": "O_kgDODxyGZg",
    "avatar_url": "https://avatars.githubusercontent.com/u/253527654?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/Health-informatics-Data-Analytics",
    "html_url": "https://github.com/Health-informatics-Data-Analytics",
    "followers_url": "https://api.github.com/users/Health-informatics-Data-Analytics/followers",
    "following_url": "https://api.github.com/users/Health-informatics-Data-Analytics/following{/other_user}",
    "gists_url": "https://api.github.com/users/Health-informatics-Data-Analytics/gists{/gist_id}",
    "starred_url": "https://api.github.com/users/Health-informatics-Data-Analytics/starred{/owner}{/repo}",
    "subscriptions_url": "https://api.github.com/users

Step 1: Access top-level fields (dict basics)

In [15]:
type(data)


dict

In [16]:
data["name"]
data["full_name"]
data["private"]
data["created_at"]


'2026-01-07T21:32:32Z'

Step 2: Navigate nested JSON (critical skill)

The "owner" field is another dictionary:

In [17]:
type(data["owner"])


dict

In [18]:
data["owner"]["login"]
data["owner"]["type"]
data["owner"]["html_url"]


'https://github.com/Health-informatics-Data-Analytics'

Step 3: Safe exploration pattern (avoid KeyErrors)

In [19]:
data.get("license")
data.get("language", "Not specified")


'Jupyter Notebook'

In [20]:
owner = data.get("owner", {})
owner.get("login")


'Health-informatics-Data-Analytics'

Step 4: Turn selected JSON into a tidy table

In [21]:
import pandas as pd

repo_summary = {
    "repo_name": data["full_name"],
    "owner": data["owner"]["login"],
    "public": not data["private"],
    "stars": data["stargazers_count"],
    "forks": data["forks_count"],
    "open_issues": data["open_issues_count"],
    "language": data.get("language"),
    "created_at": data["created_at"],
    "last_updated": data["updated_at"]
}

df = pd.DataFrame([repo_summary])
df


,repo_name,owner,public,stars,forks,open_issues,language,created_at,last_updated
0,Health-informatics-Data-Analytics/hi5304-noteb...,Health-informatics-Data-Analytics,True,1,2,0,Jupyter Notebook,2026-01-07T21:32:32Z,2026-02-20T01:48:26Z


In [22]:
import os
os.getcwd()


'/workspaces/hi5304-notebooks/learning'

In [23]:
import pandas as pd

df_patients = pd.read_csv("../data/chronic/patients.csv")

df_patients.head()


,patient_id,dob,sex,race,ethnicity,insurance,urbanicity,zip3,index_date,cohort,sdoh_risk_score_0_10
0,CDM-P200000,1948-02-23,F,Asian,Not Hispanic/Latino,Uninsured,Urban,840,2024-01-27,Both,4
1,CDM-P200001,1989-12-28,M,Unknown,Not Hispanic/Latino,Commercial,Suburban,828,2024-04-30,Diabetes only,3
2,CDM-P200002,1956-09-13,M,Unknown,Unknown,Other,Suburban,983,2024-06-27,Hypertension only,0
3,CDM-P200003,2001-08-21,F,Asian,Unknown,Other,Rural,979,2024-01-24,Hypertension only,6
4,CDM-P200004,2004-11-09,F,Black,Hispanic/Latino,Other,Rural,722,2024-02-06,Hypertension only,3


In [24]:
import pandas as pd

df_labs = pd.read_csv("../data/chronic/labs.csv")

df_labs.head()

,lab_id,patient_id,lab_date,lab_type,result_value,result_unit
0,LAB-000000001,CDM-P200000,2024-02-09,A1c,8.59,%
1,LAB-000000002,CDM-P200000,2024-05-26,A1c,7.36,%
2,LAB-000000003,CDM-P200000,2024-08-12,A1c,7.33,%
3,LAB-000000004,CDM-P200000,2024-11-13,A1c,7.60,%
4,LAB-000000005,CDM-P200000,2025-02-01,A1c,7.33,%


In [25]:
import pandas as pd

df_patients = pd.read_csv("../data/chronic/patients.csv")

df_patients.head()



,patient_id,dob,sex,race,ethnicity,insurance,urbanicity,zip3,index_date,cohort,sdoh_risk_score_0_10
0,CDM-P200000,1948-02-23,F,Asian,Not Hispanic/Latino,Uninsured,Urban,840,2024-01-27,Both,4
1,CDM-P200001,1989-12-28,M,Unknown,Not Hispanic/Latino,Commercial,Suburban,828,2024-04-30,Diabetes only,3
2,CDM-P200002,1956-09-13,M,Unknown,Unknown,Other,Suburban,983,2024-06-27,Hypertension only,0
3,CDM-P200003,2001-08-21,F,Asian,Unknown,Other,Rural,979,2024-01-24,Hypertension only,6
4,CDM-P200004,2004-11-09,F,Black,Hispanic/Latino,Other,Rural,722,2024-02-06,Hypertension only,3
